In [1]:
import pandas as pd
import numpy as np
from scipy import sparse
from recovery_model.system_builder import System

In [2]:
from itertools import product

var = {
    "flows": ("F1", "F2", "F3"),
    "products": ("P1", "P2"),
    "components": ("C1", "C2", "C3", "C4"),
    "materials": ("M1", "M2", "M3"),
    "elements": ("E1", "E2", "E3"),
}

levels = ("flows", "products", "components", "materials", "elements")
fill_idx = "\u2205"
iterables = [var[levels[0]]] + [(fill_idx,) + var[i] for i in levels[1:]]

midx = pd.DataFrame(data=list(product(*iterables)), columns=levels)

if len(var) == 5:
    lvl1_notna = midx[levels[1]] != fill_idx  # products
    lvl2_notna = midx[levels[2]] != fill_idx  # components
    lvl3_notna = midx[levels[3]] != fill_idx  # materials
    lvl4_notna = midx[levels[4]] != fill_idx  # elements
    lvl2_isna = ~lvl2_notna
    lvl3_isna = ~lvl3_notna
    case_1_2 = lvl2_notna & lvl3_isna & lvl4_notna
    case_3_4_5 = lvl1_notna & lvl2_isna & (lvl3_notna | lvl4_notna)
    rows_to_remove = midx.loc[case_1_2 | case_3_4_5].index
    midx = midx.drop(index=rows_to_remove).reset_index(drop=True)

midx = midx.replace("\u2205", "-1")
midx

,flows,products,components,materials,elements
0,F1,-1,-1,-1,-1
1,F1,-1,-1,-1,E1
2,F1,-1,-1,-1,E2
3,F1,-1,-1,-1,E3
4,F1,-1,-1,M1,-1
...,...,...,...,...,...
517,F3,P2,C4,M2,E3
518,F3,P2,C4,M3,-1
519,F3,P2,C4,M3,E1
520,F3,P2,C4,M3,E2


In [3]:
midx = midx.reset_index().set_index(list(levels)).squeeze()
midx

flows  products  components  materials  elements
F1     -1        -1          -1         -1            0
                                        E1            1
                                        E2            2
                                        E3            3
                             M1         -1            4
                                                   ... 
F3     P2        C4          M2         E3          517
                             M3         -1          518
                                        E1          519
                                        E2          520
                                        E3          521
Name: index, Length: 522, dtype: int64

In [4]:
rows = pd.IndexSlice[("F1", "-1", "-1", "-1", "-1")]
midx.loc(axis=0)[rows]

0

In [5]:
rows = (("F1", "-1", "-1", "-1", "-1"), ("F1", "P1", "C4", "-1", "-1"))
# rows = (
#     ('F1',  '-1',  '-1',  '-1', "-1"),
#     ('F1',  'P1',  'C1',  '-1', "-1"),
#     ('F1',  'P1',  'C4',  '-1', "-1")
# )

# rows = [('F1',  'P1',  'C4',  '-1', "-1")]
# rows = (('F1',  'P1',  'C4',  '-1', "-1"),)

# #! examples below don't work
# rows = (('F1',  'P1',  'C4',  '-1', "-1"))  # ! DON'T WORK
# rows = ('F1',  'P1',  'C4',  '-1', "-1")  # ! DON'T WORK

midx.loc[list(rows)]  # ! it has to be a list of tuples

flows  products  components  materials  elements
F1     -1        -1          -1         -1            0
       P1        C4          -1         -1          108
Name: index, dtype: int64

In [6]:
# rows = [('F1',  'P1',  'C4',  '-1', "-1")]

# ! PARTIALLY WORKS
rows = (("F1", "P1", "C4", "-1", "-1"),)  #! PARTIALLY WORKS
# rows = (('F1',  'P1',  'C4',  '-1', "-1"))  # ! PARTIALLY WORKS
# rows = ('F1',  'P1',  'C4',  '-1', "-1")  # ! PARTIALLY WORKS

# #! examples below don't work
# rows = (
#     ('F1',  '-1',  '-1',  '-1', "-1"),
#     ('F1',  'P1',  'C4',  '-1', "-1")
# )
# rows = (
#     ('F1',  '-1',  '-1',  '-1', "-1"),
#     ('F1',  'P1',  'C1',  '-1', "-1"),
#     ('F1',  'P1',  'C4',  '-1', "-1")
# )

midx.loc[pd.IndexSlice[rows]]

108

In [7]:
# WORKS
rows = (("F1", "P1", "C4", "M1", slice(None)),)

# ! PARTIALLY WORKS
# rows = ('F1',  'P1',  'C4',  '-1', slice(None))
# rows = ('F1',  'P1',  'C4',  'M1', slice(None))


# rows = (
#     ('F1',  '-1',  '-1',  '-1', "-1"),
#     ('F1',  'P1',  'C4',  '-1', "-1")
# )
# rows = (
#     ('F1',  '-1',  '-1',  '-1', "-1"),
#     ('F1',  'P1',  'C1',  '-1', "-1"),
#     ('F1',  'P1',  'C4',  '-1', "-1")
# )

# rows = [('F1',  'P1',  'C4',  '-1', "-1")]
# rows = (('F1',  'P1',  'C4',  '-1', "-1"),)

# #! examples below don't work

# rows = (
#     ("F1", "P1", "C4", "M1", slice(None)),
#     ("F1", "P1", "C1", "M1", slice(None)),
# )

# rows = [('F1',  'P1',  'C4',  'M1', slice(None))]
# rows = (('F1',  'P1',  'C4',  'M1', slice(None)),)

# rows = (('F1',  'P1',  'C4',  '-1', "-1"))  # ! DON'T WORK
# rows = ('F1',  'P1',  'C4',  '-1', "-1")  # ! DON'T WORK

midx.loc[pd.IndexSlice[rows]]  # ! it has to be a list of tuples

flows  products  components  materials  elements
F1     P1        C4          M1         -1          109
                                        E1          110
                                        E2          111
                                        E3          112
Name: index, dtype: int64

In [8]:
midx.iloc[[0, 2, 108]]

flows  products  components  materials  elements
F1     -1        -1          -1         -1            0
                                        E2            2
       P1        C4          -1         -1          108
Name: index, dtype: int64

In [9]:
# idx = ("F3", "P2", "C4", "M2", slice(None)) # ! don't work
# midx.loc[list(idx), :]

In [10]:
# midx.index = pd.MultiIndex.from_frame(midx)
# midx

In [11]:
# idx = ("F3", "P2", "C4", "M2", slice(None))
# # midx.index.get_indexer_for([idx])
# midx.loc[idx, :].index

In [12]:
# midx.loc[idx, :].index

In [13]:
composition_dct = {
    "url": "data/Toy_WEEE/Toy_WEEE_Composition.xlsx",
    "sheet": "Toy_WEEE_composition",
    "mapper": {
        "Flow": "flows",
        "Layer1": "products",
        "Layer2": "components",
        "Layer3": "materials",
        "Layer4": "elements",
        "Value": "data",
        "Year": "year",
        "parameterCode": "layer_code",
    },
    "data_processing": {
        "components": ("c-p",),
        "materials": ("m-p", "m-c"),
        "elements": ("e-p", "e-c", "e-m"),
    },
}

tc_dct = {
    "url": "data/Toy_WEEE/Toy_WEEE_TCs.xlsx",
    "sheet": "Toy_WEEE_TCs",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
    },
    "all_symbol": {
        "products": "P*",
        "components": "C*",
        "materials": "M*",
        "elements": "E*",
    },
    "crossboundary_inflows": ("WEEE_INFLOW",),
}

inputs_dct = {
    "url": "data/Toy_WEEE/Toy_WEEE_inputs.xlsx",
    "sheet": "Toy_WEEE_inputs",
    "unit": "tonnes",
    "mapper": {
        "input_flow": "flows",
        "Category": "products",
        "mass_(tonnes)": "data",
        "Year": "year",
    },
}

# Questions

easy case:

- `[F, P, C, M, E]`

**ambiguous case 1:** (for TC template)

- `[F, ∅, C, M, E]` vs `[F, ∅, C, M, ∅]`
  - on the left, `∅` means that there's no product layer in `F`
  - on the right, the last `∅` means all elements contained in M
  - --> `∅` can mean 2 different things!
  - replace with: `[F, ∅, C, M, E]` vs `[F, ∅, C, M, :]`?
- `[F, P, ∅, ∅, ∅]` --> `[F, P, :, :, :]`
- what about `[F, ∅, C, :, :]`?
  - does it mean `[F, ∅, C, :, :]`?
  - or does it mean `[F, :, C, :, :]`?

**ambiguous case 2:** (for TC template)

- when both `input` and `ouput` has `all` (or `:`) symbol, expanding leads to the combination of both
  - --> this creates matter


To do:

- set ones on diagonal
- set y (mass)
- solve and validate


In [14]:
weee = System(composition_dct=composition_dct, inputs_dct=inputs_dct, tc_dct=tc_dct)

input_layer1_key P* 0 Index([], dtype='int64')
input_layer1_key C* 4 Index([91, 92, 98, 99], dtype='int64')
input_layer1_key M* 2 Index([86, 93], dtype='int64')
input_layer1_key E* 0 Index([], dtype='int64')
input_layer2_key P* 0 Index([], dtype='int64')
input_layer2_key C* 0 Index([], dtype='int64')
input_layer2_key M* 0 Index([], dtype='int64')
input_layer2_key E* 0 Index([], dtype='int64')
output_layer1_key P* 0 Index([], dtype='int64')
output_layer1_key C* 0 Index([], dtype='int64')
output_layer1_key M* 20 Index([55, 56, 57, 58, 59, 60, 86, 86, 86, 86, 86, 86, 86, 93, 93, 93, 93, 93,
       93, 93],
      dtype='int64')
output_layer1_key E* 0 Index([], dtype='int64')


In [15]:
list(weee.tc_rows)

[('WEEE_generatedComplementaryMetalScrapExportedUndocumented',
  'Cat_1',
  '∅',
  '∅',
  '∅'),
 ('WEEE_generatedComplementaryMetalScrapExportedUndocumented',
  'Cat_2',
  '∅',
  '∅',
  '∅'),
 ('WEEE_generatedComplementaryMetalScrapExportedUndocumented',
  'Cat_3',
  '∅',
  '∅',
  '∅'),
 ('WEEE_generatedComplementaryMetalScrapExportedUndocumented',
  'Cat_4a',
  '∅',
  '∅',
  '∅'),
 ('WEEE_generatedComplementaryMetalScrapExportedUndocumented',
  'Cat_4b',
  '∅',
  '∅',
  '∅'),
 ('WEEE_generatedComplementaryMetalScrapExportedUndocumented',
  'Cat_5',
  '∅',
  '∅',
  '∅'),
 ('WEEE_generatedComplementaryMetalScrapExportedUndocumented',
  'Cat_6',
  '∅',
  '∅',
  '∅'),
 ('WEEE_generatedComplementaryMetalScrapExportedUndocumented',
  'Cat_6',
  '∅',
  '∅',
  '∅'),
 ('WEEE_generatedComplementaryMSW', 'Cat_2', '∅', '∅', '∅'),
 ('WEEE_generatedComplementaryMSW', 'Cat_1', '∅', '∅', '∅'),
 ('WEEE_generatedComplementaryMSW', 'Cat_3', '∅', '∅', '∅'),
 ('WEEE_generatedComplementaryMSW', 'Cat_4a', '

In [16]:
list(weee.tc_cols)

[('WEEE_INFLOW', 'Cat_1', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_2', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_3', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4a', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4b', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_5', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_2', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_1', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_3', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4a', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4b', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_5', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_1', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_2', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_3', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4a', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4b', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_5', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅

In [10]:
list(weee.index[weee.tc_rows])
# list(weee.tc_rows)

IndexError: arrays used as indices must be of integer (or boolean) type

In [5]:
list(weee.index[weee.tc_cols])
# list(weee.tc_cols)

[('WEEE_INFLOW', 'Cat_1', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_2', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_3', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4a', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4b', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_5', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_2', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_1', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_3', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4a', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4b', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_5', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_1', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_2', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_3', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4a', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4b', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_5', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅

In [8]:
pd.concat(
    {
        "inflow": weee.index[weee.tc_cols].to_frame().reset_index(drop=True),
        "outflow": weee.index[weee.tc_rows].to_frame().reset_index(drop=True),
        "TC": pd.DataFrame({"value": weee.tc_data}),
    },
    axis=1,
).to_csv("results/formatted_Toy_WEEE_TCs.csv", index=False)

In [8]:
list(weee.index[weee.input_rows])

[('WEEE_INFLOW', 'Cat_1', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_2', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_3', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4a', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4b', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_5', '∅', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', '∅', '∅', '∅')]

In [9]:
list(weee.index[weee.comp_rows])

[('WEEE_INFLOW', 'Cat_1', 'Cables', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_1', 'PCBUnspecified', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_2', 'Cables', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_2', 'PCBUnspecified', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_3', 'PCBUnspecified', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4a', 'Cables', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4a', 'PCBUnspecified', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4b', 'Cables', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_4b', 'PCBUnspecified', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_5', 'Cables', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_5', 'PCBUnspecified', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', 'Cables', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_6', 'PCBUnspecified', '∅', '∅'),
 ('WEEE_INFLOW', 'Cat_1', '∅', 'AlloyZincUnspecified', '∅'),
 ('WEEE_INFLOW', 'Cat_1', '∅', 'AluminiumAlloyUnspecified', '∅'),
 ('WEEE_INFLOW', 'Cat_1', '∅', 'Brass', '∅'),
 ('WEEE_INFLOW', 'Cat_1', '∅', 'Bronze', '∅'),
 ('WEEE_INFLOW', 'Cat_1', '∅', 'CopperAlloyUnspecified', '∅'),
 ('WEEE_INFLOW', 'Cat_1', '∅', 'NiAlloyUnspeci

In [5]:
print(weee)

{   'components': ('Cables', 'PCBUnspecified'),
    'elements': ('Al', 'Cu'),
    'flows': (   'WEEE_2RM_mechRec1Smelter_CuScrap',
                 'WEEE_2RM_mechRec2Smelter_AlScrap',
                 'WEEE_2RM_mechRec2Smelter_CuScrap',
                 'WEEE_2RM_mechRecLampsSmelter_CuScrap',
                 'WEEE_INFLOW',
                 'WEEE_PVPanels',
                 'WEEE_collected',
                 'WEEE_generatedComplementaryMSW',
                 'WEEE_generatedComplementaryMetalScrapExportedUndocumented',
                 'WEEE_generatedDedicated',
                 'WEEE_ironCircuit',
                 'WEEE_lampsDirect_(without_LED)',
                 'WEEE_largeEquipment',
                 'WEEE_screensAndMonitors',
                 'WEEE_smallEquipment',
                 'WEEE_smallIT',
                 'WEEE_temperatureExchangeEquipment',
                 'WEEE_wasteBinIncineration',
                 'WEEE_wasteBinLandfill'),
    'index': {   'components': {-1: 0, 'Cabl

In [8]:
list(weee.index)

[('WEEE_2RM_mechRec1Smelter_CuScrap', -1, -1, -1, -1),
 ('WEEE_2RM_mechRec1Smelter_CuScrap', -1, -1, -1, 'Al'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap', -1, -1, -1, 'Cu'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap', -1, -1, 'AluminiumAlloyUnspecified', -1),
 ('WEEE_2RM_mechRec1Smelter_CuScrap',
  -1,
  -1,
  'AluminiumAlloyUnspecified',
  'Al'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap',
  -1,
  -1,
  'AluminiumAlloyUnspecified',
  'Cu'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap', -1, 'Cables', -1, -1),
 ('WEEE_2RM_mechRec1Smelter_CuScrap', -1, 'Cables', -1, 'Al'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap', -1, 'Cables', -1, 'Cu'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap',
  -1,
  'Cables',
  'AluminiumAlloyUnspecified',
  -1),
 ('WEEE_2RM_mechRec1Smelter_CuScrap',
  -1,
  'Cables',
  'AluminiumAlloyUnspecified',
  'Al'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap',
  -1,
  'Cables',
  'AluminiumAlloyUnspecified',
  'Cu'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap', -1, 'PCBUnspecified', -1, -1),
 ('WEEE_2RM_mechRec1Smelter_C

In [ ]:
weee.flows

('WEEE_2RM_mechRec1Smelter_CuScrap',
 'WEEE_2RM_mechRec2Smelter_AlScrap',
 'WEEE_2RM_mechRec2Smelter_CuScrap',
 'WEEE_2RM_mechRecLampsSmelter_CuScrap',
 'WEEE_INFLOW',
 'WEEE_PVPanels',
 'WEEE_collected',
 'WEEE_generatedComplementaryMSW',
 'WEEE_generatedComplementaryMetalScrapExportedUndocumented',
 'WEEE_generatedDedicated',
 'WEEE_ironCircuit',
 'WEEE_lampsDirect_(without_LED)',
 'WEEE_largeEquipment',
 'WEEE_screensAndMonitors',
 'WEEE_smallEquipment',
 'WEEE_smallIT',
 'WEEE_temperatureExchangeEquipment',
 'WEEE_wasteBinIncineration',
 'WEEE_wasteBinLandfill')

In [ ]:
weee.products

('Cat_1', 'Cat_2')

In [ ]:
weee.materials

('AluminiumAlloyUnspecified',)

In [ ]:
weee.elements

('Al', 'Cu')

In [ ]:
print(weee)

{   'components': ('Cables', 'PCBUnspecified'),
    'elements': ('Al', 'Cu'),
    'flows': (   'WEEE_2RM_mechRec1Smelter_CuScrap',
                 'WEEE_2RM_mechRec2Smelter_AlScrap',
                 'WEEE_2RM_mechRec2Smelter_CuScrap',
                 'WEEE_2RM_mechRecLampsSmelter_CuScrap',
                 'WEEE_INFLOW',
                 'WEEE_PVPanels',
                 'WEEE_collected',
                 'WEEE_generatedComplementaryMSW',
                 'WEEE_generatedComplementaryMetalScrapExportedUndocumented',
                 'WEEE_generatedDedicated',
                 'WEEE_ironCircuit',
                 'WEEE_lampsDirect_(without_LED)',
                 'WEEE_largeEquipment',
                 'WEEE_screensAndMonitors',
                 'WEEE_smallEquipment',
                 'WEEE_smallIT',
                 'WEEE_temperatureExchangeEquipment',
                 'WEEE_wasteBinIncineration',
                 'WEEE_wasteBinLandfill'),
    'index': {   'components': {-1: 0, 'Cabl

In [ ]:
idxs = [
    ["WEEE_INFLOW", slice(None), "Cables", -1, -1],
    pd.IndexSlice["WEEE_INFLOW", "Cat_1", "Cables", :, :],
]

for idx in idxs:
    print(weee.format_index(idx), "\n")

[['WEEE_INFLOW' -1 'Cables' -1 -1]
 ['WEEE_INFLOW' 'Cat_1' 'Cables' -1 -1]
 ['WEEE_INFLOW' 'Cat_2' 'Cables' -1 -1]] 

[['WEEE_INFLOW' 'Cat_1' 'Cables' -1 -1]
 ['WEEE_INFLOW' 'Cat_1' 'Cables' -1 'Al']
 ['WEEE_INFLOW' 'Cat_1' 'Cables' -1 'Cu']
 ['WEEE_INFLOW' 'Cat_1' 'Cables' 'AluminiumAlloyUnspecified' -1]
 ['WEEE_INFLOW' 'Cat_1' 'Cables' 'AluminiumAlloyUnspecified' 'Al']
 ['WEEE_INFLOW' 'Cat_1' 'Cables' 'AluminiumAlloyUnspecified' 'Cu']] 



In [ ]:
idxs = [
    "WEEE_INFLOW",
    ["WEEE_INFLOW"],
    ("WEEE_INFLOW", "Cat_1"),
    ("WEEE_INFLOW", "Cat_1", "Cables", -1, -1),
    [("WEEE_INFLOW", "Cat_1", "Cables", -1, -1)],
    (["WEEE_INFLOW", "Cat_1", "Cables", -1, -1]),
    [["WEEE_INFLOW", "Cat_1", "Cables", -1, -1]],
    [
        ["WEEE_INFLOW", "Cat_1", "Cables", -1, -1],
        ["WEEE_INFLOW", "Cat_2", "Cables", -1, -1],
    ],
    ["WEEE_INFLOW", slice(None), "Cables", -1, -1],
    pd.IndexSlice["WEEE_INFLOW", "Cat_1", "Cables", :, :],
]

for idx in idxs:
    print(idx)
    print(weee.get_indexer(idx), "\n")

WEEE_INFLOW
[216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251
 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269] 

['WEEE_INFLOW']
[216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251
 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269] 

('WEEE_INFLOW', 'Cat_1')
[234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251] 

('WEEE_INFLOW', 'Cat_1', 'Cables', -1, -1)
[240] 

[('WEEE_INFLOW', 'Cat_1', 'Cables', -1, -1)]
[240] 

['WEEE_INFLOW', 'Cat_1', 'Cables', -1, -1]
[240] 

[['WEEE_INFLOW', 'Cat_1', 'Cables', -1, -1]]
[240] 

[['WEEE_INFLOW', 'Cat_1', 'Cables', -1, -1], ['WEEE_INFLOW', 'Cat_2', 'Cables', -1, -1]]
[240 258] 

['WEEE_INFLOW', slice(None, None, None), 'Cables', -1, -1]
[222 240 258] 

('WEEE_INFLOW', 'Cat_1', 'Cables', slice(No

In [32]:
int_idx = [240, 241, 242, 243, 244, 245]
list(weee.index_iloc(int_idx))

[('WEEE_2RM_mechRec1Smelter_CuScrap', 'Cat_3', 'Cables', '∅', '∅'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap', 'Cat_3', 'Cables', '∅', 'Al'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap', 'Cat_3', 'Cables', '∅', 'Cu'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap',
  'Cat_3',
  'Cables',
  'AgAlloyUnspecified',
  '∅'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap',
  'Cat_3',
  'Cables',
  'AgAlloyUnspecified',
  'Al'),
 ('WEEE_2RM_mechRec1Smelter_CuScrap',
  'Cat_3',
  'Cables',
  'AgAlloyUnspecified',
  'Cu')]

In [33]:
weee.index.shape

(10944,)

In [42]:
random_iloc = np.random.randint(0, weee.index.shape, 100)
random_idx = weee.index[random_iloc]
random_iloc

array([   68,  7524,  6275, 10181,  4873,  2111,  3013, 10379,  6028,
        1870, 10481,  5015,  8380, 10761,  1722, 10392,  6897,  1813,
        8177,  8447,  9208,   448,  7564, 10002,  9988,  7335,  3425,
        9556,  3929, 10732,  5164,  2067,  2403,  5778,  4868,  9716,
        6352,  9790,  7000, 10413,  4471,  6137, 10273,  2510,  4146,
        7720,  2038,  2223,  6404,   768,  3191,  7669,  1255,  1213,
        2450,  6064,  1158,  5628,  9188,  5800,   601,  3206,  5297,
        4993,  1777,  8931,  1325, 10043,  1552,  6507,  9532,  8963,
        4172,  9250,  5950,  3511,  8723,  5684,  4396,  1971,  6391,
         729,  2221,  4819,   372,  2925,  7588,  4221,  3276,  7636,
        4810,  2044,  3667, 10444,  3314,  6416,  5060,  9654,  4536,
        4096])

In [50]:
%%timeit
weee.get_indexer(random_idx)

4.29 ms ± 118 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [51]:
%%timeit
weee.index.get_indexer(random_idx)

346 µs ± 18.7 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [52]:
%%timeit
weee.index.get_indexer(weee.format_index(random_idx))

5.93 ms ± 222 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [53]:
weee.format_index(random_idx)

array([['WEEE_2RM_mechRec1Smelter_CuScrap', '∅', 'PCBUnspecified',
        'CopperAlloyUnspecified', 'Cu'],
       ['WEEE_screensAndMonitors', '∅', 'Cables', 'Brass', '∅'],
       ['WEEE_ironCircuit', 'Cat_6', '∅', 'AluminiumAlloyUnspecified',
        'Cu'],
       ['WEEE_wasteBinIncineration', 'Cat_4b', 'Cables',
        'AgAlloyUnspecified', 'Cu'],
       ['WEEE_generatedComplementaryMetalScrapExportedUndocumented',
        'Cat_3', 'PCBUnspecified', '∅', 'Al'],
       ['WEEE_2RM_mechRecLampsSmelter_CuScrap', 'Cat_4b', '∅',
        'NiAlloyUnspecified', 'Cu'],
       ['WEEE_PVPanels', 'Cat_1', 'PCBUnspecified', 'Brass', 'Al'],
       ['WEEE_wasteBinLandfill', '∅', '∅', 'AluminiumAlloyUnspecified',
        'Cu'],
       ['WEEE_ironCircuit', 'Cat_3', 'PCBUnspecified',
        'AgAlloyUnspecified', 'Al'],
       ['WEEE_2RM_mechRecLampsSmelter_CuScrap', 'Cat_1',
        'PCBUnspecified', 'NiAlloyUnspecified', 'Al'],
       ['WEEE_wasteBinLandfill', 'Cat_1', 'Cables', 'Bronze', 'Cu'],
   

In [10]:
mi = pd.MultiIndex.from_arrays([list("abb"), list("def"), list("xyz")], names=["col1", "col2", "col3"])
mi

MultiIndex([('a', 'd', 'x'),
            ('b', 'e', 'y'),
            ('b', 'f', 'z')],
           names=['col1', 'col2', 'col3'])

In [11]:
keys = [1, 2]
mi.to_frame().iloc[keys].index

MultiIndex([('b', 'e', 'y'),
            ('b', 'f', 'z')],
           names=['col1', 'col2', 'col3'])

In [12]:
mi[keys]

MultiIndex([('b', 'e', 'y'),
            ('b', 'f', 'z')],
           names=['col1', 'col2', 'col3'])

---


In [12]:
solution = weee.solve()
solution.to_csv("results/solution.csv")
solution

flows                             products  components      materials                  elements
WEEE_2RM_mechRec1Smelter_CuScrap  -1        -1              -1                         -1          0.0
                                                                                       Al          0.0
                                                                                       Cu          0.0
                                                            AluminiumAlloyUnspecified  -1          0.0
                                                                                       Al          0.0
                                                                                                  ... 
WEEE_wasteBinLandfill             Cat_2     PCBUnspecified  -1                         Al          0.0
                                                                                       Cu          0.0
                                                            AluminiumAlloyUnspec

In [13]:
np.vstack(
    [
        weee.tcs.tocoo().row,
        weee.tcs.tocoo().col,
        # weee.tcs.tocoo().data,
    ]
)

array([[   0,    1,    2, ..., 1023, 1024, 1025],
       [   0,    1,    2, ..., 1023, 1024, 1025]], dtype=int32)

In [14]:
tcs = weee.tcs.copy()
temp_tc = pd.DataFrame(tcs.toarray(), index=weee.index, columns=weee.index)


xbrdy_inflow = tc_dct["crossboundary_inflows"][0]
# for cat in ("Cat_1", "Cat_2", "Cat_3", "Cat_4a", "Cat_4b", "Cat_5", "Cat_6"):
#     idx_ax0 = pd.IndexSlice[xbrdy_inflow, cat, -1, -1]
#     idx_ax1 = pd.IndexSlice["products", cat]
#     temp_tc.loc[idx_ax0, idx_ax1] = 1

idx = pd.IndexSlice[xbrdy_inflow, :, :, :]
temp_tc.loc[idx, idx].to_csv("results/temp_tc.csv")

In [15]:
y = weee.y.copy()
temp_y = pd.DataFrame(y.toarray(), index=weee.index, columns=weee.columns)


temp_y.loc[idx].to_csv("results/temp_y.csv")

AttributeError: 'System' object has no attribute 'columns'

In [ ]:
(temp_tc * temp_y).loc[pd.IndexSlice[xbrdy_inflow, :, :, :]].to_csv("tc_dot_y.csv")

In [ ]:
A = temp_tc.loc[pd.IndexSlice[xbrdy_inflow, :, :, :]]
A.to_numpy().shape

In [ ]:
Y = temp_y.loc[pd.IndexSlice[xbrdy_inflow, :, :, :]]
Y.to_numpy().shape

In [ ]:
file = "data/test/TC Matrix Building v2.xlsx"
tcs = pd.read_excel(file, sheet_name="tc", header=None).values
y = pd.read_excel(file, sheet_name="y", header=None).values

In [ ]:
from scipy.sparse.linalg import spsolve
from scipy.sparse import csr_matrix, csc_matrix

A = csr_matrix(tcs)
Y = csr_matrix(y)
spsolve(A, Y)

In [ ]:
Afull = csr_matrix(tcs)
Afull.nonzero()

In [ ]:
np.vstack([Afull.nonzero(), Afull.data]).T

In [ ]:
def squarify(mat):
    max_dim = max(mat.shape)
    dims = np.array(mat.shape)
    new_shape = np.full(dims.shape, fill_value=max_dim)
    new_mat = np.zeros(new_shape)
    new_mat[: mat.shape[0], : mat.shape[1]] = mat
    return new_mat


new_A = squarify(A.to_numpy())

In [ ]:
res = new_A * Y.to_numpy()
pd.DataFrame(res, index=A.index, columns=Y.columns).to_csv("temp_res.csv")

In [ ]:
xbrdy_inflow = tc_dct["crossboundary_inflows"][0]
pp = pd.DataFrame(np.zeros(shape=(len(weee.index), len(weee.columns))), index=weee.index, columns=weee.columns)

prod = "Cat_1"
comp = slice(None)
mat = -1
idx_ax0 = pd.IndexSlice[xbrdy_inflow, prod, comp, mat]
idx_ax1 = pd.IndexSlice["products", prod]

pp.loc[idx_ax0, idx_ax1]

In [ ]:
def pruned(coo_mat, idxs, cols):
    csr_mat = coo_mat.tocsr()

    rows, _ = csr_mat.nonzero()
    unique_rows = np.sort(np.unique(rows))
    data = csr_mat[unique_rows, :].toarray()
    return pd.DataFrame(data=data, index=idxs[unique_rows], columns=cols)

In [ ]:
pruned(weee.y, weee.index, weee.columns).to_csv("test.csv")

In [ ]:
# i = weee.materials[0]
# i
rw = tuple([[-1, i, -1] for i in weee.materials])
cl = tuple([("materials", i) for i in weee.materials])
rw

<hr>

# Sparse matrices


In [6]:
from scipy.sparse import coo_array, coo_matrix, linalg

In [7]:
rows1 = np.random.randint(6, size=10)
cols1 = np.random.randint(6, size=10)
data1 = np.random.randint(10, size=10)

print(np.array([rows1, cols1, data1]))

coo_arr1 = sparse.coo_array((data1, (rows1, cols1)), shape=(10, 6))
print(coo_arr1.shape)
csr_arr1 = coo_arr1.tocsr()
coo_arr1.toarray()

[[3 5 2 4 4 0 3 0 0 3]
 [2 4 2 2 3 1 4 5 5 3]
 [6 5 1 1 8 9 9 5 9 1]]
(10, 6)


array([[ 0,  9,  0,  0,  0, 14],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  1,  0,  0,  0],
       [ 0,  0,  6,  1,  9,  0],
       [ 0,  0,  1,  8,  0,  0],
       [ 0,  0,  0,  0,  5,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0]])

In [8]:
rows2 = np.random.randint(6, size=10)
cols2 = np.random.randint(6, size=10)
data2 = np.random.randint(10, size=10)

print(np.array([rows2, cols2, data2]))

coo_arr2 = sparse.coo_array((data2, (rows2, cols2)), shape=(10, 6))
csr_arr2 = coo_arr2.tocsr()
coo_arr2.toarray()

[[4 0 4 5 3 0 2 0 4 4]
 [3 1 5 3 5 3 0 2 0 2]
 [0 0 2 8 8 5 7 3 0 8]]


array([[0, 0, 3, 5, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [7, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 8],
       [0, 0, 8, 0, 0, 2],
       [0, 0, 0, 8, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]])

In [9]:
(csr_arr1 @ csr_arr2.T).toarray()

array([[  0,   0,   0, 112,  28,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  3,   0,   0,   0,   8,   0,   0,   0,   0,   0],
       [ 23,   0,   0,   0,  48,   8,   0,   0,   0,   0],
       [ 43,   0,   0,   0,   8,  64,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0]])

In [18]:
type(csr_arr1)

scipy.sparse._arrays.csr_array

In [10]:
type(sparse.vstack([coo_arr1, coo_arr2]))

scipy.sparse._coo.coo_matrix

In [11]:
sparse.vstack([coo_arr1, coo_arr2]).toarray()

array([[ 0,  9,  0,  0,  0, 14],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  1,  0,  0,  0],
       [ 0,  0,  6,  1,  9,  0],
       [ 0,  0,  1,  8,  0,  0],
       [ 0,  0,  0,  0,  5,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  3,  5,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 7,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  8],
       [ 0,  0,  8,  0,  0,  2],
       [ 0,  0,  0,  8,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0]])

In [12]:
csr_arr1[[0], :].toarray()

array([[ 0,  9,  0,  0,  0, 14]])

In [13]:
seq = [csr_arr1] * 5
sparse.vstack(seq)

<50x6 sparse matrix of type '<class 'numpy.int64'>'
	with 45 stored elements in Compressed Sparse Row format>

In [14]:
rows3 = np.random.randint(6, size=10)
cols3 = np.random.randint(6, size=10)
data3 = np.random.randint(10, size=10)

print(np.array([rows3, cols3, data3]))

coo_mat = sparse.coo_matrix((data2, (rows2, cols2)), shape=(10, 6))
csr_mat = coo_mat.tocsr()
csr_mat.toarray()

[[3 1 1 4 5 2 0 1 0 5]
 [1 0 2 5 3 2 5 0 2 0]
 [4 5 3 0 0 2 4 9 5 2]]


array([[0, 0, 3, 5, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [7, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 8],
       [0, 0, 8, 0, 0, 2],
       [0, 0, 0, 8, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]])

In [15]:
csr_mat

<10x6 sparse matrix of type '<class 'numpy.int64'>'
	with 10 stored elements in Compressed Sparse Row format>

In [16]:
A = np.ones((5, 5))
A

array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]])

In [17]:
S = sparse.csr_matrix(A)
S

<5x5 sparse matrix of type '<class 'numpy.float64'>'
	with 25 stored elements in Compressed Sparse Row format>